# Instantiating pgvector

In [ ]:
# This is to make it so that we cna run asyncio event loop in jupyter notebook.
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from pypdf import PdfReader
from llama_index.core import Document

reader = PdfReader("data/Transformer_Architecture_Big_Picture_and_Details.pdf")
documents = [
    Document(
        text=page.extract_text(),
        metadata={"page_label": str(i + 1)},
        doc_id=f"transformer_paper_page_{i + 1}",
    )
    for i, page in enumerate(reader.pages)
]

In [ ]:
from llama_index.core import  Settings, VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.postgres import PGVectorStore

Settings.embed_model = HuggingFaceEmbedding(model_name= "intfloat/multilingual-e5-small")
Settings.llm = GoogleGenAI(model="gemini-3.8-flash")

vector_store = PGVectorStore.from_params(
  database="ragdb",
  host="localhost",
  password="ragpass",
  port=5432,
  user="raguser",
  table_name="transformer_paper",
  embed_dim=384,
  hnsw_kwargs={
    "hnsw_m": 16,
    "hnsw_ef_construction": 64,
    "hnsw_ef_search": 40,
    "hnsw_dict_method": "vector_cosine_ops",
  }
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

for doc in documents:
    vector_store.delete(doc.doc_id)
    
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

query_engine = index.as_query_engine(similarity_tok_k=4)
response = query_engine.query("What is the role of self attention in the transformer architecture?")

print(response)
for node in response.source_nodes:
  print(node.node.metadata, node.score)


In [ ]:
len(reader.pages)

In [ ]:
prompts = query_engine.get_prompts()
for name, template in prompts.items():
    print(f"--- {name} ---")
    print(template.get_template())
    print()

In [ ]:
Settings.llm.metadata

In [ ]:
qa_template = query_engine.get_prompts()["response_synthesizer:chat_content_qa_template"]
with open("qa_template.txt", "w") as f:
    f.write(qa_template.get_template())

In [ ]:
from llama_index.core.query_engine import CitationQueryEngine

citation_query_engine = CitationQueryEngine.from_args(
    index,
    similarity_top_k=4,
    citation_chunk_size=512,
)

response = citation_query_engine.query("What is the role of self-attention in the transformer architecture?")
print(response)
for i, node in enumerate(response.source_nodes):
    print(f"[{i + 1}] page {node.node.metadata.get('page_label')}: {node.node.text[:80]}...")